# Step 3 – Vasastan heat-production optimisation
**Sweden · SE3 operating prices 2019 · five Step 2 producers · no storage**

Reads the existing Step 2 consumer/producer CSVs and the same demand profiles/scaling as Steps 1–2. Does not regenerate the network. Network losses and pipe constraints are integrated in Step 4, not in this aggregate Step 3 model.

Inputs: `data/el_spot_price.csv`, `data/step2/twn_data/`, and `data/step2/demand_selected.csv` (preferred) or the original demand Excel/cache. Outputs: `data/step3/`, `results/step3/`.

**Provisional assumptions:** assumed site capacity limits; see section 5

Confirmed scenario: identical HP technology data and unchanged unrestricted investment bounds, as requested. The supplied Step 2 demand_selected.csv is included.


## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import oemof.solph as solph
from helpers import (DATA, RESULTS, SOLVER, YEAR, INTEREST_RATE, epc,
                     demand_scaled, DEMAND_SCALE, PRODUCERS, CONSUMERS,
                     build_and_solve_heat_system, summarize_units,
                     heat_dispatch, plot_dispatch, UNIT_COLORS)
IN, OUT = DATA / "step3", RESULTS / "step3"
IN.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Aggregated heat demand of the ten Step 2 consumers
The profile names are read from the Step 2 network (`consumers.csv`, column `profile`), so Step 2 and Step 3 always use the same consumers. The profiles are multiplied by `DEMAND_SCALE` = 0.4 (`helpers.py` section 0), exactly as in Steps 1–2; see Step 1, section 1 for why.

**Unit conversion:** demand profiles are in kW (hourly) → divide by 1000 → **MW**. With hourly time steps, 1 MW for one hour = 1 MWh, so all oemof energy results are in MWh and costs in €/MWh.

In [ ]:
consumers = pd.read_csv(DATA / "step2/twn_data/consumers.csv")
producers = pd.read_csv(DATA / "step2/twn_data/producers.csv")
expected_c = CONSUMERS[CONSUMERS.step <= 2].reset_index(drop=True)
expected_p = PRODUCERS[PRODUCERS.step <= 2].reset_index(drop=True)
assert len(consumers) == 10 and len(producers) == 5
assert consumers["profile"].tolist() == expected_c["profile"].tolist(), "Consumer profiles differ from helpers.py"
assert producers["id"].tolist() == list(range(5)), "Unexpected Step 2 producer IDs"
for actual, expected in [(consumers, expected_c), (producers, expected_p)]:
    assert np.allclose(actual[["lat", "lon"]], expected[["lat", "lon"]], atol=1e-6, rtol=0), "Network coordinates differ from helpers.py"
STEP2_LOADS = consumers["profile"].tolist()
selected = DATA / "step2/demand_selected.csv"
if selected.exists():
    demand = pd.read_csv(selected, index_col=0)[expected_c["key"].tolist()]
    print("Using saved Step 2 hourly demand; already scaled, no second scaling.")
else:
    demand = demand_scaled(STEP2_LOADS)
    demand.columns = expected_c["key"].tolist()
    print("Using original demand shapes x DEMAND_SCALE =", DEMAND_SCALE)
assert len(demand) == 8760 and np.isfinite(demand.to_numpy()).all()
assert (demand.to_numpy() >= 0).all()
assert np.allclose(demand.max().to_numpy(), consumers["P_heat_max"], atol=0.011, rtol=0), "Demand peaks differ from Step 2; check cache/scaling"
heat_demand_MW = demand.sum(axis=1) / 1000
assert heat_demand_MW.max() > 0
print(f"Peak {heat_demand_MW.max():.3f} MW | annual {heat_demand_MW.sum():.1f} MWh")
display(expected_p[["key", "name", "technology", "step"]])

## 2. Operating prices: SE3 2019 and purchased biomethane
Electricity: uploaded OPSD series `SE_3_price_day_ahead`, EUR/MWh electric input, 2019. Source: https://data.open-power-system-data.org/time_series/2020-10-06 . Spot prices only: grid charges, electricity taxes and VAT are omitted.

Purchased biomethane: **102.4 EUR/MWh fuel**, constant reference assumption, from the supplied EU source https://eur-lex.europa.eu/eli/C/2024/1921/oj . This is not a verified Stockholm delivered-gas tariff. Network charges are omitted. `gas price` is retained as the existing helper/Step 4–5 column name.

UTC timestamps are checked for consecutive hourly intervals and converted to **fixed CET (UTC+1)** for a regular 2019 calendar. This avoids duplicate/missing daylight-saving clock hours. Demand profiles are aligned sequentially to these 8760 hours; their original timezone is not provided and this alignment is an explicit assumption.

**Reviewed data gap:** two prices are missing at UTC 2019-03-31 02:00 and 03:00. Linear interpolation fills these two hours only. Original input is unchanged; replacements are exported to `electricity_price_gap_log.csv`. These are estimates, not observed market prices.


In [ ]:
# Read the hourly electricity prices.
raw_prices = pd.read_csv(DATA / "el_spot_price.csv")

# Parse timestamps as UTC.
utc = pd.DatetimeIndex(
    pd.to_datetime(raw_prices["utc_timestamp"], utc=True)
)

# Verify that the file contains 8760 unique, chronological hourly timestamps.
assert len(utc) == 8760 and utc.is_unique and utc.is_monotonic_increasing
assert ((utc[1:] - utc[:-1]) == pd.Timedelta(hours=1)).all(), \
    "Time steps are not exactly one hour"

# Convert to a fixed CET calendar (UTC+1), without daylight-saving changes.
model_index = utc.tz_convert(None) + pd.Timedelta(hours=1)
expected_index = pd.date_range(
    f"{YEAR}-01-01", periods=8760, freq="h"
)

assert (model_index == expected_index).all(), \
    "Electricity prices do not cover the expected year in fixed CET"

# Extract electricity prices in EUR/MWh and identify missing values.
el = pd.to_numeric(
    raw_prices["SE_3_price_day_ahead"], errors="raise"
)
missing = el.isna()

assert missing.sum() <= 2, \
    "More than two electricity prices are missing; inspect the source file"
assert not missing.iloc[0] and not missing.iloc[-1], \
    "Cannot interpolate missing prices at the beginning or end"

# Fill the two reviewed missing prices using linear interpolation.
# These replacements are estimates, not observed market prices.
filled = el.interpolate(
    method="linear", limit=2, limit_area="inside"
)
assert np.isfinite(filled.to_numpy()).all(), \
    "Missing or invalid electricity prices remain"

# Save an audit log of the estimated prices; keep the original CSV unchanged.
price_audit = pd.DataFrame({
    "date": expected_index[missing.to_numpy()],
    "utc_timestamp": raw_prices.loc[missing, "utc_timestamp"].to_numpy(),
    "original_price": el[missing].to_numpy(),
    "replacement_eur_per_MWh": filled[missing].to_numpy(),
    "method": "linear interpolation between adjacent valid hourly values",
})
price_audit.to_csv(IN / "electricity_price_gap_log.csv", index=False)

print("Interpolated missing electricity prices:", int(missing.sum()))
display(price_audit)

# Assume a constant biomethane price of 102.4 EUR/MWh of fuel input.
# Keep the existing "gas price" column name for compatibility with the model.
BIOGAS_PRICE = 102.4
prices = pd.DataFrame({
    "el_spot_price": filled.to_numpy(),
    "gas price": BIOGAS_PRICE,
}, index=expected_index)

# Align demand profiles sequentially with the same hourly calendar.
# This assumes the demand rows correspond to the fixed-CET year.
heat_demand_MW.index = expected_index
demand.index = expected_index
prices.index.name = "date"

# Display summary statistics for the input prices.
display(prices.describe().round(2))

## 3. Two recovered-heat sources; no solar thermal
P0 recovers station ventilation heat; P4 recovers supermarket refrigeration heat. Both are represented as heat pumps with separate free heat-source buses. No measured hourly availability profile or source limit is available.

No solar profile is generated. `solar_cf = 0` is retained only as a compatibility column for the later input-file format; no solar component is built.

In [ ]:
solar_cf = pd.Series(0.0, index=prices.index)
print("P0 station heat pump + P4 supermarket heat pump; no solar producer.")

## 4. Assemble and save the oemof input table
One table with everything time-dependent, hourly, in MW and €/MWh. Saved to `data/step3/oemof_input.csv` → Step 4 adds the network losses to it.

In [ ]:
data = pd.DataFrame({
    "heat demand":   heat_demand_MW.values,           # MW
    "gas price":     prices["gas price"].values,      # €/MWh fuel
    "el_spot_price": prices["el_spot_price"].values,  # €/MWh electricity
    "solar_cf":      solar_cf.values,                 # -  (0..1)
}, index=prices.index)
data.index.name = "date"
data.to_csv(IN / "oemof_input.csv")
data.describe().round(3)

## 5. Technology and carrier data, with explicit boundaries
IDs: P0 station HP; P1 pellet boiler; P2 electric boiler; P3 biomethane boiler; P4 supermarket HP.

DEA catalogue: https://ens.dk/en/analyses-and-statistics/technology-data-generation-electricity-and-district-heating . Values are transcribed from the supplied catalogue. 

- Both HPs: waste-heat 1 MW reference, COP 3.10, CAPEX 1.13 MEUR/MW_heat, variable O&M 3.07 EUR/MWh_heat, lifetime 25 years. Applying this to station air and supermarket heat is an assumption. 

- Pellets: heat-only wood-pellet boiler with flue-gas condensation, annual efficiency 1.01, CAPEX 0.75 MEUR/MW_heat, total variable O&M 2.08, lifetime 25.
- Electric: **small** proxy (5 MW reference), efficiency 0.99, CAPEX 0.15 MEUR/MW_heat, total variable O&M 0.95, lifetime 20. This proxy is extrapolated to this small network.
- Gas: Natural Gas DH Only proxy adapted to purchased biomethane, annual efficiency 1.03, CAPEX 0.06 MEUR/MW_heat, total variable O&M 1.17, lifetime 25.

**Costs:**  Annual costs include annualised CAPEX, fuel/main electricity and variable O&M. Total catalogue variable O&M retains catalogue auxiliary-electricity costs; auxiliary electricity is not an additional input flow, avoiding separate duplicate charging. Main electricity/fuel input is priced separately. Thus auxiliary costs do not vary with SE3 prices.

**Investment capacity limits:** The maximum installed heat capacities are
assumed to be 0.20 MW for each heat pump and 0.15 MW for each boiler.
These limits define a demand-based sizing scenario, using the peak
consumer heat demand of approximately 0.613 MW as a reference.

The heat pumps are intended primarily for base-load production, while
the pellet boiler provides intermediate load and the electric and
biomethane boilers provide peak-load or reserve capacity. These intended
roles do not impose dispatch rules: the optimiser determines installed
capacities and hourly operation from the economic parameters.

The combined upper capacity limit is 0.85 MW. These values are engineering
assumptions, not catalogue limits or verified local resource capacities.
Available waste heat, electricity and gas connections have not been
verified. The capacity margin does not guarantee operation during a
producer outage.

**Emissions:** electricity 0.026 tCO2e/MWh_input, Swedish 2019 mix (national, not SE3), https://www.energimyndigheten.se/klimat/hallbarhetskriterier/fragor-och-svar-om-hallbarhetskriterier/ . Pellets/biomethane 0 fossil combustion CO2 with biogenic CO2 excluded, https://www.naturvardsverket.se/amnesomraden/klimatomstallningen/omraden/klimatet-och-energin/hallbar-bioenergi/ . This is not a complete life-cycle GHG comparison; fuel supply emissions/CH4/N2O are omitted. Biomethane procurement is assumed, not a claim of 100% physical biogas in the Stockholm network.

Pellet fuel price: 294 SEK/MWh, EN0307 2019 refined wood fuels for heating plants; proxy includes pellets, briquettes and powder. Divide by 10.589 SEK/EUR = 27.76 EUR/MWh. https://pxexternal.energimyndigheten.se/pxweb/sv/Energimyndighetens_statistikdatabas/Energimyndighetens_statistikdatabas__Officiell_energistatistik__Tradbransle_och_torvpriser/

In [ ]:
DEA = "https://ens.dk/en/analyses-and-statistics/technology-data-generation-electricity-and-district-heating"
tech = pd.DataFrame([
    # unit, input, efficiency, capex, total variable O&M, max heat MW, lifetime, type, source
    ["heat pump station", "electricity", 3.10, 1_130_000, 3.07, np.nan, 25, "heat_pump", "DEA waste heat 1 MW 2025; station proxy; " + DEA],
    ["biomass boiler", "biomass", 1.01, 750_000, 2.08, np.nan, 25, "boiler", "DEA Wood Pellets HOP 2020; " + DEA],
    ["electric boiler", "electricity", 0.99, 150_000, 0.95, np.nan, 20, "boiler", "DEA Electric boiler small 2025; extrapolated reference; " + DEA],
    ["gas boiler", "gas", 1.03, 60_000, 1.17, np.nan, 25, "boiler", "DEA Natural Gas DH Only 2020; biomethane proxy; " + DEA],
    ["heat pump supermarket", "electricity", 3.10, 1_130_000, 3.07, np.nan, 25, "heat_pump", "DEA waste heat 1 MW 2025; supermarket proxy; " + DEA],
], columns=["unit", "input", "efficiency", "capex_eur_per_MW", "var_om_eur_per_MWh",
            "max_capacity_MW", "lifetime_yr", "technology_type", "source"]).set_index("unit")
tech["DHNx producer"] = expected_p["key"].tolist()
tech["producer name"] = expected_p["name"].tolist()
assert tech.index.is_unique
assert tech["DHNx producer"].tolist() == ["P0", "P1", "P2", "P3", "P4"]
carriers = pd.DataFrame([
    ["gas", np.nan, 0.0, "Purchased 100% biomethane assumption; 102.4 EUR/MWh in timeseries; biogenic CO2 excluded; https://eur-lex.europa.eu/eli/C/2024/1921/oj"],
    ["biomass", 27.76, 0.0, "EN0307 2019 refined wood fuels to heating plants 294 SEK/MWh /10.589; pellets proxy; biogenic CO2 excluded"],
    ["electricity", np.nan, 0.026, "OPSD SE3 2019 timeseries; EF Swedish national 2019 mix, Energimyndigheten; tCO2e/MWh"],
], columns=["carrier", "price_eur_per_MWh", "ef_tCO2_per_MWh", "source"]).set_index("carrier")
mean_prices = {"gas": data["gas price"].mean(), "biomass": carriers.loc["biomass", "price_eur_per_MWh"],
               "electricity": data["el_spot_price"].mean()}

# Assumed site capacity limits [MW_heat].
# Demand-based sizing scenario; local resource availability is not verified.
capacity_limits = {
    "heat pump station": 0.20,
    "heat pump supermarket": 0.20,
    "biomass boiler": 0.15,
    "electric boiler": 0.15,
    "gas boiler": 0.15,
}

tech["max_capacity_MW"] = tech.index.map(capacity_limits)
report = tech.copy()
report["mean input price [EUR/MWh_in]"] = tech["input"].map(mean_prices)
report["mean fuel cost [EUR/MWh_heat]"] = report["mean input price [EUR/MWh_in]"] / tech["efficiency"]
report["model emissions [t/MWh_heat]"] = tech["input"].map(carriers["ef_tCO2_per_MWh"]) / tech["efficiency"]
report["annualised capex [EUR/MW/a]"] = [epc(c, i=INTEREST_RATE, n=n) for c, n in zip(tech.capex_eur_per_MW, tech.lifetime_yr)]
print("ASSUMPTION: HP capacity limits 0.20 MW each; boiler limits 0.15 MW each.")
tech.to_csv(IN / "technology_parameters.csv")
carriers.to_csv(IN / "carriers.csv")
report.to_csv(IN / "parameter_report.csv")
display(report.round(3))

## 6. Optimisation problem
Units are the five producer rows above; t = 1…8760, hourly intervals, no storage.

Decision variables: installed heat capacity P_u [MW], hourly heat Q_u,t [MW], and fuel/electricity input F_u,t [MW].

Minimise annualised CAPEX + hourly fuel/electricity costs + variable O&M per MWh of heat. Step 5 can additionally price the chosen emissions metric.

Constraints: sum_u Q_u,t = D_t every hour; Q_u,t <= P_u; Q = efficiency × input; P_u <= specified investment bound (NaN unrestricted). For each HP, electricity = Q/COP and recovered heat = Q×(COP−1)/COP.

`helpers.py` constructs each technology row dynamically, with separate HP source buses. Step 4–5 must load the exported tables and keep these same assumptions.

In [ ]:
model, results, parts = build_and_solve_heat_system(data, tech, carriers, demand_col="heat demand", i=INTEREST_RATE)

sr = model.solver_results
status = (sr.solver.termination_condition if hasattr(sr, "solver")          # CBC: pyomo SolverResults
          else sr.get("termination_condition", "n/a"))                   # HiGHS: plain dict
print("Solver:", SOLVER, "| termination condition:", status)
print(f"Objective (total annual cost): {model.objective():,.0f} EUR/a")

## 7. Verify that the heat demand is satisfied
Hourly heat balance: sum of all unit outputs must equal the demand in **every** hour.

In [ ]:
dispatch = heat_dispatch(results, parts)          # MW per unit + demand
units = [c for c in dispatch.columns if c != "demand"]
imbalance = dispatch[units].sum(axis=1) - data["heat demand"]
print(f"Max hourly imbalance: {imbalance.abs().max():.2e} MW")
print(f"Heat produced {dispatch[units].sum().sum():,.1f} MWh vs demand {data['heat demand'].sum():,.1f} MWh")
assert imbalance.abs().max() < 1e-4, "Demand not met!"
print("✅ Demand met in all 8760 hours")

## 8. Results per unit
Capacity, heat share, full-load hours, hours at maximum capacity, and the cost breakdown. The total must equal the objective (consistency check).

In [ ]:
summary = summarize_units(results, parts, tech, carriers)
print(f"Objective {model.objective():,.0f}  vs  sum of unit costs {summary.loc['TOTAL', 'total cost [EUR]']:,.0f}")
summary.round(1)
assert np.isclose(model.objective(), summary.loc["TOTAL", "total cost [EUR]"], rtol=1e-6, atol=0.1), "Cost report differs from objective"

cost_report = summary.copy()
cost_report["operating cost [EUR/year]"] = (
    cost_report["fuel/electricity cost [EUR]"]
    + cost_report["var O&M cost [EUR]"]
)

display(cost_report[[
    "capacity [MW]",
    "share of heat [%]",
    "operating cost [EUR/year]",
    "total cost [EUR]",
]].round(2))

## 9. Dispatch over the year
Stacked heat production (daily means) and duration curves. ✏️ Describe which units supply base, mid and peak load and why (merit order = marginal cost; investment cost vs. full-load hours).

In [ ]:
plot_dispatch(dispatch, "Heat production by unit – Step 3")
plt.savefig(OUT / "dispatch.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
week = dispatch["2019-01-14":"2019-01-20"]
ax.stackplot(week.index, *[week[u] for u in units], labels=units, colors=[UNIT_COLORS[u] for u in units])
ax.plot(week.index, week["demand"], "k", lw=1, label="demand")
ax.set_title("Hourly dispatch, one winter week"); ax.set_ylabel("MW"); ax.legend(ncol=6, loc="upper center")
plt.tight_layout(); plt.show()

## 10. When do units reach their maximum capacity?
Count of hours per month in which each unit runs at ≥ 99.9 % of its installed capacity, and these hours marked on the demand curve – relate them to the demand profile (cold periods? source limits?).

In [ ]:
caps = summary.loc[units, "capacity [MW]"]
at_cap = pd.DataFrame({u: dispatch[u] >= 0.999 * caps[u] for u in units if caps[u] > 1e-6})
hours_at_cap_by_month = at_cap.groupby(at_cap.index.month).sum()
hours_at_cap_by_month.index.name = "month"
display(hours_at_cap_by_month)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dispatch.index, dispatch["demand"], color="lightgrey", lw=0.6, label="demand")
for u in at_cap.columns:
    hrs = at_cap.index[at_cap[u]]
    ax.scatter(hrs, dispatch.loc[hrs, "demand"], s=3, color=UNIT_COLORS[u], label=f"{u} at max")
ax.set_ylabel("Demand [MW]"); ax.set_title("Hours in which a unit is at its maximum capacity"); ax.legend(markerscale=4, ncol=3)
plt.tight_layout(); plt.savefig(OUT / "hours_at_capacity.png", dpi=150); plt.show()

## 11. Operating cost per unit – which unit contributes most to system cost?

In [ ]:
cost_cols = ["annualised capex [EUR]", "fuel/electricity cost [EUR]", "var O&M cost [EUR]"]
summary.loc[units, cost_cols].plot.bar(stacked=True, figsize=(10, 4), title="Annual cost per unit – Step 3")
plt.ylabel("EUR/a"); plt.tight_layout(); plt.savefig(OUT / "costs_per_unit.png", dpi=150); plt.show()
# Annualised costs already use each technology's own lifetime.
print(f"Annualised system cost per delivered heat: {summary.loc['TOTAL', 'total cost [EUR]'] / data['heat demand'].sum():.2f} EUR/MWh")

## 12. Save results (used by Step 4 and 5)

In [ ]:
summary.to_csv(OUT / "unit_summary.csv")
dispatch.to_csv(OUT / "dispatch.csv")
hours_at_cap_by_month.to_csv(OUT / "hours_at_capacity_by_month.csv")
print("Saved to", OUT)

## 13. Questions to answer
✏️

1. **Which generating units supply the majority of the heat demand? How is the load distributed among the units?** *(section 8 share of heat, section 9 plots)*

The two heat pumps supply 94.76% of the annual heat demand: 44.91% from the station heat pump and 49.85% from the supermarket heat pump. The electric boiler supplies 4.68%, the pellet boiler 0.40%, and the biomethane boiler 0.15%.

The heat pumps dominate production because of their low operating costs. The boilers provide supplementary heat. Since both heat pumps have identical technology parameters, their individual production shares are not unique.

2. **Are there periods when any unit reaches its maximum defined capacity? Which units, and when?** *(section 10 – relate to demand peaks / heat-source availability / the `max_capacity_MW` limits)*

The heat pumps reach their installed capacities throughout the year, most frequently during winter. In January, the station heat pump operates at capacity for 683 hours and the supermarket heat pump for 690 hours.
The pellet boiler reaches capacity mainly during winter: 597 hours in January, 561 in February and 418 in December. The electric boiler reaches capacity during shorter periods, including 81 hours in February. The biomethane boiler reaches capacity for only three hours, two in January and one in February.

3. **What are the total operational costs for each unit, and which unit contributes most to system costs?** *(section 11)*

Operating costs consist of fuel/electricity costs plus variable O&M. Annualised investment costs are reported separately, and fixed O&M is excluded.
The total annual system cost, including annualised investment, is approximately €83,183, equivalent to €29.00/MWh of delivered heat.

The supermarket heat pump contributes the most to both operating and total system costs in this solution, mainly because it supplies the largest heat share. A high annual cost does not necessarily mean a high cost per MWh: the heat pumps incur most of the expenditure because they produce most of the heat.

4. Report the formulation (section 6), the solution status (6.1) and the demand check (7), and justify every 🔧 parameter with a source.

The model minimises annualised investment costs, fuel/electricity costs and variable O&M. It optimises installed heat capacities and hourly dispatch, subject to technology conversion efficiencies, capacity limits and an hourly heat balance. No thermal storage is included.

HiGHS reports an optimal solution. Annual heat production and demand both equal 2,868.3 MWh, and demand is satisfied in all 8,760 hours. The maximum hourly imbalance is approximately 5.55 × 10⁻¹⁷ MW.

Technology parameters come from the DEA catalogue; electricity prices from OPSD; fuel prices and emission factors from the sources listed in section 5. The capacity limits are demand-based engineering assumptions, rather than verified site capacities.